In [2]:
%load_ext autoreload
%autoreload 2
import os, sys

import re
import numpy as np
import pandas as pd


from inDecay import PATH,my_utils
os.chdir(PATH.main_dir)
from inDecay.models import Topk_Event_Overlapping

In [27]:
from tqdm.auto import tqdm
import pickle as pkl
from scipy.stats import pearsonr

pj = os.path.join

In [4]:
from qrguide import analysis_fn, transformation

In [5]:
## read Test set 
experiments =["ST_June_2017_BOB_LV7A_DPI7", "ST_June_2017_BOB_LV7B_DPI7" ,
      "ST_June_2017_CHO_LV7A_DPI7", "ST_June_2017_CHO_LV7B_DPI7",
      "ST_June_2017_E14TG2A_LV7A_DPI7", "ST_June_2017_E14TG2A_LV7B_DPI7",
      "ST_June_2017_HAP1_LV7A_DPI7", "ST_June_2017_HAP1_LV7B_DPI7",
      "ST_June_2017_K562_800x_LV7A_DPI7", "ST_June_2017_K562_800x_LV7B_DPI7",]

celltypes = [exp.split("_")[3] for exp in experiments if 'LV7A' in exp]
celltype_rename = ['iPSC', 'CHO', 'mESC', 'HAP1', 'K562']

In [6]:
celltypes, celltype_rename

(['BOB', 'CHO', 'E14TG2A', 'HAP1', 'K562'],
 ['iPSC', 'CHO', 'mESC', 'HAP1', 'K562'])

In [19]:
pj = os.path.join

def read_pkl(path):
    with open(path, 'rb') as f:
        Y = pkl.load(f)
    f.close()
    return Y

def evalute_fn(Y_true_path, Y_pred_path):
    Y_pred =read_pkl(Y_pred_path)
    Y = read_pkl(Y_true_path)

    eval_json = analysis_fn.assessment_recipe_forecast(Y, Y_pred)
    eval_json.update(analysis_fn.assessment_recipe_IDL_forecast(Y, Y_pred))

    eval_df = pd.json_normalize(eval_json)
    return eval_df

def get_ratios(Y_true_path, Y_pred_path):

    pred_lookup =read_pkl(Y_pred_path)
    Y_lookup = read_pkl(Y_true_path)


    ratio_json = []
    for oligo, Y in Y_lookup.items():
        Y = Y.T

        pred = pred_lookup[oligo]
        Indel = Y[[0],:]
        y = Y[[1],:].astype("float32")

        # frameshift
        y_fs, pred_fs = analysis_fn.forecast_frameshift(y, pred, Indel)
        y_dr, pred_dr = analysis_fn.forecast_delratio(y, pred, Indel)

        ratio_json.append(
            {'Gene':oligo, "Rep1_frameshift":y_fs, "Pred_frameshift":pred_fs, "Rep_delratio":y_dr, "Pred_delratio":pred_dr}
        )
    
    return pd.json_normalize(ratio_json)

def ratio_error(row, ratio):
    error = row[f"Rep1_{ratio}"] - row[f'Pred_{ratio}']
    return np.abs(error).item()

# All_df['Abs. Err. Frameshift Ratio'] = All_df.apply(ratio_error , axis=1, args=('frameshift',))
# All_df['Abs. Err. Deletion Ratio'] = All_df.apply(ratio_error , axis=1, args=('delratio',))

# Key function for auto evalution

In [74]:
def find_pkl_and_eval(data_archive):
    """
    given the data_archive name, auto find 
    """
    archive_folder = f"{PATH.main_dir}/pl_trainer_log/{data_archive}_featv5_ST_DeepDecay_identity_C1"


    Cells = ['iPSC', 'mESC']
    Fix_settings = ['None', 'del_regressor[:1]', 'del_regressor[:2]']

    # 5 fold validation

    perform = []
    ratio_df = []

    for cell in Cells:
        for fix in Fix_settings:
            # for k_index in tqdm(range(5), desc=f"{cell} {fix}", leave=False):
            for k_index in range(5):

                def annotate_df(df):
                    df['kfold_index'] = k_index
                    df['celltype'] = cell
                    df['fix_setting'] = fix
                
                # the directory name
                second_save_path = f"{cell}_featv5_pretrained_{fix}_5fold_{k_index}"

                # auto find pkl files
                Y_true = pj(archive_folder, second_save_path, "ForeCast_TestY.pkl")
                Y_baseline = pj(archive_folder, second_save_path, "Pretrained_Baseline_TestPred.pkl")
                Y_pred_path = my_utils.find_ckpt(pj(archive_folder, second_save_path, "lightning_logs"))
                Y_pred_path = Y_pred_path.replace(".ckpt", "TestPred.pkl")


                # get all the evaluation metrics            
                df_k = evalute_fn(Y_true, Y_pred_path)
                df_baseline_k = evalute_fn(Y_true, Y_baseline)
                df_baseline_k['celltype'] = cell

                # get frameshift / del ratio for each genes
                ratio_df_k = get_ratios(Y_true, Y_pred_path)
                ratio_df_baseline_k = get_ratios(Y_true, Y_baseline)
                ratio_df_baseline_k['celltype'] = cell

                ratio_df_k['Baseline_frameshift'] = ratio_df_baseline_k['Pred_frameshift']
                ratio_df_k['Baseline_delratio'] = ratio_df_baseline_k['Pred_delratio']

                # annotate
                for df in [df_k,df_baseline_k,ratio_df_k]:
                    annotate_df(df)

                df_baseline_k['fix_setting'] = 'baseline'

                perform.append(df_k)
                perform.append(df_baseline_k)
                ratio_df.append(ratio_df_k)
            
    perform  = pd.concat(perform)
    ratio_df = pd.concat(ratio_df)
    

    return perform, ratio_df 

# evaluate a single data archive

In [63]:
# use os.walk to return directories only
Sanger_archive_ls = [folder.split("_featv5")[0] for folder in os.listdir(PATH.pth_dir) if folder.startswith('Sanger')]
Sanger_archive_ls[:3], Sanger_archive_ls[-3:]

(['Sanger_decr2_invalidlabel_LowCount',
  'Sanger_decr2_icecosine_LowCount',
  'Sanger_icecosine_invalidlabel_NoICE_LowCount'],
 ['Sanger_decr2_NoICE',
  'Sanger_decr2_icecosine_invalidlabel_NoICE_LowCount',
  'Sanger_decr2_invalidlabel_NoICE'])

## find Ytrue and Ypred pkl

In [55]:
from tqdm.auto import tqdm

In [75]:
# which takes around 14 seconds
perform, ratio_df = find_pkl_and_eval("Sanger_icecosine_NoICE_LowCount")

perform_mean = perform.groupby(["celltype", "fix_setting"]).agg("mean").reset_index()
# perform_baseline = perform_baseline.groupby(["celltype"]).agg("mean")

In [76]:
perform_mean

,celltype,fix_setting,KL divergence,Top5 events recall,Top10 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top5_IDL,Top10_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,kfold_index
0,iPSC,None,6.660377,1.990476,3.842857,0.268557,2.457143,4.366667,1.064075,2.795238,4.747619,0.216027,0.137346,0.384702,2.0
1,iPSC,baseline,6.409124,2.152381,3.533333,0.272580,2.547619,4.090476,0.905942,2.766667,5.247619,0.201564,0.507818,0.396067,2.0
2,iPSC,del_regressor[:1],6.439976,2.214286,3.814286,0.235479,2.547619,4.300000,0.944304,2.761905,5.004762,0.204407,0.256439,0.400032,2.0
3,iPSC,del_regressor[:2],6.386343,2.176190,3.700000,0.319968,2.609524,4.223810,1.036967,2.738095,4.809524,0.226129,0.237239,0.401158,2.0
4,mESC,None,6.332869,2.200000,3.938095,0.235442,2.633333,4.214286,0.964964,2.876190,4.857143,0.207571,0.300489,0.403518,2.0
5,mESC,baseline,6.172438,2.066667,3.542857,0.219291,2.485714,4.028571,0.927449,2.861905,5.090476,0.202476,0.564744,0.399390,2.0
6,mESC,del_regressor[:1],6.264026,2.295238,3.995238,0.215621,2.661905,4.304762,0.946219,2.966667,4.742857,0.201111,0.295009,0.406500,2.0
7,mESC,del_regressor[:2],6.372691,2.228571,3.900000,0.265417,2.533333,4.328571,0.976650,2.876190,4.857143,0.213698,0.230014,0.397416,2.0


In [72]:
ratio_df.head()

,Gene,Rep1_frameshift,Pred_frameshift,Rep_delratio,Pred_delratio,Baseline_frameshift,Baseline_delratio,kfold_index,celltype,fix_setting
0,H2K1-sg2,0.795224,0.811379,0.638206,0.700373,0.822357,0.924117,0,iPSC,None
1,Rag1-sg1-4,0.888889,0.930068,0.888889,0.295772,0.897242,0.875335,0,iPSC,None
2,sry,1.000000,0.857497,1.000000,0.851633,0.911124,0.968655,0,iPSC,None
3,ZNRF4-sg2,0.865988,0.946290,0.896298,0.304272,0.860473,0.716432,0,iPSC,None
4,TPO,0.926199,0.813741,0.734317,0.783664,0.650772,0.902255,0,iPSC,None


# iterate all data archive

In [78]:
perform_all = []
ratio_df_all = []

for data_archive in tqdm(Sanger_archive_ls, desc='iter data', leave=False):

    try:
        perform,  ratio_df = find_pkl_and_eval(data_archive)

        perform_mean = perform.groupby(["celltype", "fix_setting"]).agg("mean").reset_index()

        perform['data_archive'] = data_archive
        ratio_df['data_archive'] = data_archive

        
        perform_all.append(perform)
        ratio_df_all.append(ratio_df)
        
    except FileNotFoundError:
        print(data_archive, 'file missing')

perform_all = pd.concat(perform_all)
ratio_df_all = pd.concat(ratio_df_all)

iter data:   0%|          | 0/31 [00:00<?, ?it/s]

Sanger_invalidlabel_NoICE file missing
Sanger_decr2_invalidlabel_NoICE_LowCount file missing
Sanger_decr2_NoICE_LowCount file missing
Sanger_decr2_icecosine_NoICE file missing
Sanger_icecosine_invalidlabel_LowCount file missing
Sanger_LowCount file missing
Sanger_icecosine_invalidlabel_NoICE file missing
Sanger_decr2_icecosine_NoICE_LowCount file missing
Sanger_decr2_LowCount file missing
Sanger_invalidlabel file missing
Sanger_decr2_icecosine_invalidlabel_NoICE file missing
Sanger_decr2_NoICE file missing
Sanger_decr2_icecosine_invalidlabel_NoICE_LowCount file missing
Sanger_decr2_invalidlabel_NoICE file missing


In [79]:
perform_all.data_archive.nunique()

17

In [82]:
print(perform_all.shape)
perform_all.to_csv("results/Sanger_benchmark/17archive_36gens_5fold_perform_Aug29.csv", index=False)

(1020, 16)


In [81]:

ratio_df_all.

,KL divergence,Top5 events recall,Top10 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top5_IDL,Top10_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,kfold_index,celltype,fix_setting,data_archive
0,7.269389,2.428571,4.142857,0.193850,2.857143,4.428571,1.038780,2.857143,4.428571,0.266880,0.212597,0.381129,0,iPSC,None,Sanger_decr2_invalidlabel_LowCount
0,6.380760,1.714286,3.857143,0.303504,2.571429,4.285714,1.130262,2.714286,4.857143,0.277955,0.292373,0.366353,0,iPSC,baseline,Sanger_decr2_invalidlabel_LowCount
0,5.000759,2.500000,4.166667,0.706851,2.666667,4.833333,0.596068,2.833333,4.666667,0.208174,0.353847,0.418395,1,iPSC,None,Sanger_decr2_invalidlabel_LowCount
0,4.770844,2.666667,3.833333,0.749582,2.666667,4.333333,0.528581,2.666667,5.166667,0.180497,0.783388,0.437203,1,iPSC,baseline,Sanger_decr2_invalidlabel_LowCount
0,4.863572,2.500000,3.833333,0.017050,3.166667,4.500000,0.710403,3.500000,5.666667,0.114074,0.025661,0.462792,2,iPSC,None,Sanger_decr2_invalidlabel_LowCount
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,7.356689,2.142857,3.142857,0.346108,2.428571,4.000000,0.874551,2.857143,4.857143,0.252800,0.787140,0.367004,2,mESC,baseline,Sanger_NoICE
0,5.895892,2.142857,3.714286,0.041576,2.571429,4.000000,0.934691,2.857143,4.857143,0.141322,0.023996,0.402091,3,mESC,del_regressor[:2],Sanger_NoICE
0,7.121242,1.857143,3.571429,0.044095,2.142857,3.714286,1.026063,2.571429,5.285714,0.153375,0.140591,0.416385,3,mESC,baseline,Sanger_NoICE
0,6.658991,2.000000,4.142857,0.033635,2.428571,4.142857,1.395489,2.714286,4.142857,0.288379,0.003739,0.321797,4,mESC,del_regressor[:2],Sanger_NoICE


In [83]:
print(ratio_df_all.shape)
ratio_df_all.to_csv("results/Sanger_benchmark/17archive_36gens_5fold_fsdel_ratio_Aug29.csv", index=False)

(3372, 11)


In [84]:
ratio_df_all

,Gene,Rep1_frameshift,Pred_frameshift,Rep_delratio,Pred_delratio,Baseline_frameshift,Baseline_delratio,kfold_index,celltype,fix_setting,data_archive
0,H2K1-sg2,0.795224,0.794266,0.638206,0.790406,0.822357,0.924117,0,iPSC,None,Sanger_decr2_invalidlabel_LowCount
1,ELF5,0.691265,0.880456,0.825352,0.861068,0.841576,0.904241,0,iPSC,None,Sanger_decr2_invalidlabel_LowCount
2,Dcbld2,0.837419,0.875123,0.834839,0.814572,0.841556,0.860084,0,iPSC,None,Sanger_decr2_invalidlabel_LowCount
3,EPHA2,1.000000,0.868390,0.868690,0.798424,0.905816,0.757946,0,iPSC,None,Sanger_decr2_invalidlabel_LowCount
4,ULK4-sg2,1.000000,0.981457,1.000000,0.124164,0.898701,0.506941,0,iPSC,None,Sanger_decr2_invalidlabel_LowCount
...,...,...,...,...,...,...,...,...,...,...,...
2,H2K1-sg3,1.000000,0.834016,1.000000,0.908446,0.855429,0.985199,4,mESC,del_regressor[:2],Sanger_NoICE
3,H2AB1-sg1,0.935765,0.949624,0.927829,0.893685,0.943858,0.948792,4,mESC,del_regressor[:2],Sanger_NoICE
4,sry,1.000000,0.914626,1.000000,0.885749,0.919287,0.970922,4,mESC,del_regressor[:2],Sanger_NoICE
5,Kat2a,0.875336,0.925314,0.920157,0.952597,0.918028,0.944309,4,mESC,del_regressor[:2],Sanger_NoICE
